# ASRA-Security — Cloud GPT-OSS Test

**Purpose:** Run `aicomp test` with `--agent gpt_oss` on Kaggle T4 (Mac cannot infer 20B).

**Settings:** GPU T4, **Internet ON**, private notebook.

**Not a competition submit kernel** — no inference server, no submission quota used.

See repo: `private/theory/cloud-gpt-oss-testing.md`

In [ ]:
import os
import site
import subprocess
import sys

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "aicomp-sdk>=2.0.0",
        "torch",
        "transformers",
        "accelerate",
        "huggingface_hub",
    ]
)
# Ensure console scripts (aicomp) are on PATH for shell cells
user_base = site.getuserbase()
if user_base:
    bindir = os.path.join(user_base, "bin")
    os.environ["PATH"] = bindir + os.pathsep + os.environ.get("PATH", "")
print("Dependencies installed")
print("aicomp:", subprocess.run(["which", "aicomp"], capture_output=True, text=True).stdout.strip() or "use python -m aicomp_sdk.cli.main")

In [ ]:
import shutil
import subprocess
from pathlib import Path

WORK = Path("/kaggle/working/asra-security")
REPO = "https://github.com/ilakkmanoharan/ASRA-Security.git"

if WORK.exists():
    subprocess.run(["git", "-C", str(WORK), "pull", "--ff-only"], check=False)
else:
    subprocess.check_call(["git", "clone", "--depth", "1", REPO, str(WORK)])

attack_src = WORK / "attack.py"
if not attack_src.is_file():
    # Fallback: bundled copy next to notebook on manual upload
    for candidate in Path("/kaggle/input").rglob("attack.py"):
        WORK.mkdir(parents=True, exist_ok=True)
        shutil.copy(candidate, attack_src)
        print(f"Copied attack.py from {candidate}")
        break

if not attack_src.is_file():
    raise FileNotFoundError("attack.py not found — check GitHub clone or upload attack.py as dataset")

%cd {WORK}
print(f"Using {attack_src} ({attack_src.stat().st_size} bytes)")

In [ ]:
# Optional: HuggingFace login for gated models (Gemma). GPT-OSS is usually public.
import os

hf_token = os.environ.get("HF_TOKEN", "").strip()
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient

        hf_token = UserSecretsClient().get_secret("HF_TOKEN").strip()
    except Exception as exc:
        print(f"No HF_TOKEN secret ({exc}); skipping HF login (GPT-OSS may still work)")

if hf_token:
    from huggingface_hub import login

    login(token=hf_token)
    print("HuggingFace login OK")
else:
    print("Skipped HuggingFace login")

In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "aicomp_sdk.cli.main", "validate", "attack.py"])

In [ ]:
# Phase 2 — GPT-OSS proxy (adjust budget as needed)
import subprocess
import sys

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "aicomp_sdk.cli.main",
        "test",
        "attack.py",
        "--track",
        "redteam",
        "--budget-s",
        "300",
        "--agent",
        "gpt_oss",
        "--env",
        "gym",
        "--verbose",
        "--agent-debug-jsonl",
        "/kaggle/working/gpt-oss-debug.jsonl",
        "--name",
        "cloud-gpt-oss-300s",
    ]
)

In [ ]:
# Phase 3 — Gemma proxy (optional; requires HF_TOKEN Kaggle secret if gated)
import subprocess
import sys

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "aicomp_sdk.cli.main",
        "test",
        "attack.py",
        "--track",
        "redteam",
        "--budget-s",
        "300",
        "--agent",
        "gemma",
        "--env",
        "gym",
        "--verbose",
        "--agent-debug-jsonl",
        "/kaggle/working/gemma-debug.jsonl",
        "--name",
        "cloud-gemma-300s",
    ]
)

In [ ]:
import json
import shutil
from pathlib import Path

history_dir = Path(".aicomp/history")
runs = sorted(history_dir.glob("*.json"), key=lambda p: p.stat().st_mtime)
print("Recent runs:")
for path in runs[-5:]:
    data = json.loads(path.read_text())
    attack = data.get("attack", {})
    print(
        f"  {path.name}: agent={data.get('agent_selection')} "
        f"score={attack.get('score')} findings={attack.get('findings_count')} "
        f"time={attack.get('time_taken'):.1f}s"
    )

# Copy artifacts to /kaggle/working for download
out = Path("/kaggle/working/test-artifacts")
out.mkdir(exist_ok=True)
for name in ["gpt-oss-debug.jsonl", "gemma-debug.jsonl"]:
    src = Path(f"/kaggle/working/{name}")
    if src.is_file():
        shutil.copy(src, out / name)
for path in runs[-2:]:
    shutil.copy(path, out / path.name)
print(f"Artifacts in {out}")